# Generador de musica en ABC notation

Proyecto final de Deep Learning. Entrenamos un **mini-GPT** (Transformer decoder a nivel de caracter) para generar melodias en [ABC notation](https://abcnotation.com/), y lo comparamos contra un **GRU** como baseline.

- **Datos:** tunes de [thesession.org](https://github.com/adactio/thesession-data)
- **Tarea:** modelado de lenguaje causal (predecir el siguiente caracter)
- **Comparacion:** GPT vs GRU en loss, perplexity y % de tunes musicalmente validos (music21)

## 1. Setup

En el H200 (entorno `usfq`) casi todo ya esta instalado; `music21` suele faltar.

In [ ]:
#%pip install -q music21 transformers lightning torchmetrics

import os
import sys
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from transformers import GPT2Config, GPT2LMHeadModel, get_cosine_schedule_with_warmup

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# importamos nuestras utilidades de datos (viven en ../src)
sys.path.append(os.path.abspath("../src"))
sys.path.append(os.path.abspath("src"))
import abc_utils as A

L.seed_everything(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 2. Datos

Descargamos los tunes de thesession, armamos el corpus (cada tune con cabecera `M:`/`K:`) y lo tokenizamos a nivel de caracter.

In [ ]:
csv_path = A.download_tunes()
corpus, bloques = A.build_corpus(csv_path)
tok = A.CharTokenizer(corpus)

print("tunes usados:", len(bloques))
print("caracteres del corpus:", len(corpus))
print("tamano del vocabulario:", tok.vocab_size)
print("\n--- ejemplo de tune ---")
print(bloques[0])

In [ ]:
# ===================== HIPERPARAMETROS =====================
# Contexto: cuantos caracteres ve el modelo hacia atras. 256 alcanza para
# cubrir un tune completo (cabecera + varias frases), lo que ayuda a que
# respete tonalidad y estructura (AABB).
BLOCK_SIZE = 256
BATCH_SIZE = 64

# mini-GPT (queda en ~6-7M params: mas chico que ResNet18 y BERT)
GPT_N_LAYER = 6
GPT_N_HEAD = 8
GPT_N_EMBD = 256
GPT_DROPOUT = 0.1     # dropout para que no memorice y genere tunes nuevos
GPT_LR = 3e-4         # lr tipico y estable para entrenar un GPT pequeno

# baseline GRU
GRU_EMB = 128
GRU_HIDDEN = 256
GRU_LAYERS = 2
GRU_DROPOUT = 0.2
GRU_LR = 2e-3         # las RNN toleran un lr mas alto

# entrenamiento por steps (el corpus tiene millones de ventanas, no usamos epochs)
MAX_STEPS = 5000
WARMUP_STEPS = 200
VAL_INTERVAL = 500

# generacion
TEMPERATURE = 0.9
TOP_K = 50
MAX_NEW_TOKENS = 300
# ===========================================================

In [ ]:
class CharDataset(Dataset):
    """Ventanas deslizantes de caracteres. x es el trozo e y es el mismo
    trozo corrido una posicion (el siguiente caracter a predecir)."""

    def __init__(self, data, block_size, stride=1):
        self.data = data
        self.block_size = block_size
        self.stride = stride
        self.n = max(0, (len(data) - block_size - 1) // stride)

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        j = i * self.stride
        x = torch.tensor(self.data[j:j + self.block_size], dtype=torch.long)
        y = torch.tensor(self.data[j + 1:j + 1 + self.block_size], dtype=torch.long)
        return x, y


class ABCDataModule(L.LightningDataModule):
    def __init__(self, ids, block_size, batch_size):
        super().__init__()
        self.ids = ids
        self.block_size = block_size
        self.batch_size = batch_size

    def setup(self, stage=None):
        n = len(self.ids)
        n_train, n_val = int(0.8 * n), int(0.1 * n)
        train_ids = self.ids[:n_train]
        val_ids = self.ids[n_train:n_train + n_val]
        test_ids = self.ids[n_train + n_val:]
        # train con stride 1 (todas las ventanas); val/test sin solape para que sean chicos
        self.train_ds = CharDataset(train_ids, self.block_size, stride=1)
        self.val_ds = CharDataset(val_ids, self.block_size, stride=self.block_size)
        self.test_ds = CharDataset(test_ids, self.block_size, stride=self.block_size)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=2, drop_last=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size, num_workers=2)

    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.batch_size, num_workers=2)


ids = tok.encode(corpus)
dm = ABCDataModule(ids, BLOCK_SIZE, BATCH_SIZE)
dm.setup()
print("ventanas train:", len(dm.train_ds), "| val:", len(dm.val_ds), "| test:", len(dm.test_ds))

## 3. Modelo mini-GPT

Usamos `GPT2LMHeadModel` de HuggingFace (corre sobre PyTorch) con una config pequena. Calculamos la loss de forma manual con `cross_entropy` sobre `(x, y)`, igual que haremos con el GRU, para que la comparacion sea justa.

In [ ]:
def sample_next(logits, temperature, top_k):
    """Elige el siguiente caracter: baja/sube la aleatoriedad con la
    temperatura y limita a los top_k candidatos mas probables."""
    logits = logits / temperature
    if top_k:
        v, _ = torch.topk(logits, top_k)
        logits[logits < v[:, [-1]]] = -float("inf")
    probs = F.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)


class GPTLightning(L.LightningModule):
    def __init__(self, vocab_size, block_size, n_layer, n_head, n_embd, dropout, lr, max_steps, warmup_steps):
        super().__init__()
        self.save_hyperparameters()
        config = GPT2Config(
            vocab_size=vocab_size,
            n_positions=block_size,
            n_embd=n_embd,
            n_layer=n_layer,
            n_head=n_head,
            resid_pdrop=dropout,
            embd_pdrop=dropout,
            attn_pdrop=dropout,
        )
        self.model = GPT2LMHeadModel(config)

    def forward(self, x):
        return self.model(input_ids=x).logits

    def _loss(self, batch):
        x, y = batch
        logits = self(x)
        return F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))

    def training_step(self, batch, _):
        loss = self._loss(batch)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        self.log("val_loss", self._loss(batch), prog_bar=True)

    def test_step(self, batch, _):
        self.log("test_loss", self._loss(batch))

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, betas=(0.9, 0.95), weight_decay=0.01)
        sched = get_cosine_schedule_with_warmup(opt, self.hparams.warmup_steps, self.hparams.max_steps)
        return [opt], [{"scheduler": sched, "interval": "step"}]

    @torch.no_grad()
    def generate(self, tok, prompt=A.TUNE_START, max_new_tokens=300, temperature=0.9, top_k=50):
        self.eval()
        ids = torch.tensor([tok.encode(prompt)], device=self.device)
        for _ in range(max_new_tokens):
            ids_cond = ids[:, -self.hparams.block_size:]
            logits = self(ids_cond)[:, -1, :]
            nxt = sample_next(logits, temperature, top_k)
            ids = torch.cat([ids, nxt], dim=1)
        return tok.decode(ids[0].tolist())


gpt = GPTLightning(tok.vocab_size, BLOCK_SIZE, GPT_N_LAYER, GPT_N_HEAD, GPT_N_EMBD,
                   GPT_DROPOUT, GPT_LR, MAX_STEPS, WARMUP_STEPS)
n_params = sum(p.numel() for p in gpt.parameters())
print(f"parametros del mini-GPT: {n_params/1e6:.2f}M")

## 4. Baseline GRU

Mismo objetivo (predecir el siguiente caracter) pero con una RNN recurrente en vez de atencion. Al generar arrastramos el estado oculto `h`, que es lo natural en una GRU.

In [ ]:
class GRULightning(L.LightningModule):
    def __init__(self, vocab_size, emb_dim, hidden, num_layers, dropout, lr):
        super().__init__()
        self.save_hyperparameters()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.gru = nn.GRU(emb_dim, hidden, num_layers, batch_first=True, dropout=dropout)
        self.head = nn.Linear(hidden, vocab_size)

    def forward(self, x, h=None):
        out, h = self.gru(self.emb(x), h)
        return self.head(out), h

    def _loss(self, batch):
        x, y = batch
        logits, _ = self(x)
        return F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))

    def training_step(self, batch, _):
        loss = self._loss(batch)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        self.log("val_loss", self._loss(batch), prog_bar=True)

    def test_step(self, batch, _):
        self.log("test_loss", self._loss(batch))

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

    @torch.no_grad()
    def generate(self, tok, prompt=A.TUNE_START, max_new_tokens=300, temperature=0.9, top_k=50):
        self.eval()
        ids = tok.encode(prompt)
        x = torch.tensor([ids], device=self.device)
        logits, h = self(x)              # procesamos el prompt de una vez
        salida = list(ids)
        for _ in range(max_new_tokens):
            nxt = sample_next(logits[:, -1, :], temperature, top_k)
            salida.append(int(nxt.item()))
            logits, h = self(nxt, h)     # seguimos arrastrando el estado h
        return tok.decode(salida)


gru = GRULightning(tok.vocab_size, GRU_EMB, GRU_HIDDEN, GRU_LAYERS, GRU_DROPOUT, GRU_LR)
print(f"parametros del GRU: {sum(p.numel() for p in gru.parameters())/1e6:.2f}M")

## 5. Entrenamiento

Entrenamos por `MAX_STEPS` (no por epochs, porque hay millones de ventanas). Guardamos el mejor checkpoint segun `val_loss`.

In [ ]:
def entrenar(modelo, nombre):
    ckpt = ModelCheckpoint(monitor="val_loss", mode="min", save_top_k=1, dirpath=f"checkpoints/{nombre}")
    logger = CSVLogger("logs", name=nombre)
    trainer = L.Trainer(
        max_steps=MAX_STEPS,
        val_check_interval=VAL_INTERVAL,
        check_val_every_n_epoch=None,
        limit_val_batches=50,
        callbacks=[ckpt],
        logger=logger,
        accelerator="auto",
        devices="auto",
        log_every_n_steps=50,
        enable_progress_bar=True,
    )
    trainer.fit(modelo, dm)
    return trainer, ckpt


trainer_gpt, ckpt_gpt = entrenar(gpt, "gpt")

In [ ]:
trainer_gru, ckpt_gru = entrenar(gru, "gru")

In [ ]:
def curva(ax, nombre):
    m = pd.read_csv(f"logs/{nombre}/version_0/metrics.csv")
    tr = m.dropna(subset=["train_loss"])
    va = m.dropna(subset=["val_loss"])
    ax.plot(tr["step"], tr["train_loss"], label="train")
    ax.plot(va["step"], va["val_loss"], label="val")
    ax.set_title(nombre.upper())
    ax.set_xlabel("step")
    ax.set_ylabel("loss")
    ax.legend()


fig, axs = plt.subplots(1, 2, figsize=(11, 4))
curva(axs[0], "gpt")
curva(axs[1], "gru")
plt.tight_layout()
plt.savefig("../informe/figs/loss.png", dpi=120)
plt.show()

## 6. Evaluacion y comparacion

Medimos `test_loss` y la **perplexity** (`exp(loss)`, mientras mas baja mejor) con el mejor checkpoint de cada modelo.

In [ ]:
resultados = {}

for nombre, modelo, trainer, ckpt in [("GPT", gpt, trainer_gpt, ckpt_gpt),
                                      ("GRU", gru, trainer_gru, ckpt_gru)]:
    r = trainer.test(modelo, dm, ckpt_path="best")[0]
    loss = r["test_loss"]
    resultados[nombre] = {"test_loss": loss, "perplexity": math.exp(loss)}

tabla = pd.DataFrame(resultados).T
print(tabla)

## 7. Generacion de tunes

Cargamos el mejor checkpoint de cada modelo y generamos algunas melodias.

In [ ]:
gpt_best = GPTLightning.load_from_checkpoint(ckpt_gpt.best_model_path).to(device)
gru_best = GRULightning.load_from_checkpoint(ckpt_gru.best_model_path).to(device)

print("=== tune generado por el GPT ===")
salida_gpt = gpt_best.generate(tok, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE, top_k=TOP_K)
print(A.extract_first_tune(salida_gpt))

print("\n=== tune generado por el GRU ===")
salida_gru = gru_best.generate(tok, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE, top_k=TOP_K)
print(A.extract_first_tune(salida_gru))

## 8. Validez musical y export

Generamos varios tunes por modelo y medimos que porcentaje es **ABC bien formado** (que music21 logra parsear). Guardamos un ejemplo del GPT como MIDI y como partitura.

In [ ]:
N_GEN = 100

def porcentaje_validos(modelo):
    validos, tunes = 0, []
    for _ in range(N_GEN):
        t = A.extract_first_tune(modelo.generate(tok, max_new_tokens=MAX_NEW_TOKENS,
                                                  temperature=TEMPERATURE, top_k=TOP_K))
        tunes.append(t)
        if A.is_valid_abc(t):
            validos += 1
    return 100.0 * validos / N_GEN, tunes

pct_gpt, tunes_gpt = porcentaje_validos(gpt_best)
pct_gru, tunes_gru = porcentaje_validos(gru_best)
resultados["GPT"]["validos_%"] = pct_gpt
resultados["GRU"]["validos_%"] = pct_gru

tabla = pd.DataFrame(resultados).T
print(tabla)
tabla.to_csv("../informe/figs/resultados.csv")

In [ ]:
from music21 import converter

# tomamos el primer tune valido generado por el GPT y lo exportamos
ejemplo = next((t for t in tunes_gpt if A.is_valid_abc(t)), tunes_gpt[0])
print(ejemplo)

with open("../samples/ejemplo_gpt.abc", "w") as f:
    f.write(ejemplo)

pieza = converter.parse(ejemplo, format="abc")
pieza.write("midi", "../samples/ejemplo_gpt.mid")

# la partitura en imagen necesita MuseScore/LilyPond; si no esta, seguimos igual
try:
    pieza.write("musicxml.png", "../informe/figs/partitura.png")
    print("partitura guardada")
except Exception as e:
    print("no se pudo renderizar la partitura (falta MuseScore/LilyPond):", e)

## 9. Ablation de temperatura

La temperatura controla cuanta aleatoriedad hay al generar. Vemos como afecta el % de tunes validos del GPT: muy baja repite, muy alta se vuelve ruido.

In [ ]:
temps = [0.5, 0.7, 0.9, 1.1, 1.3]
pct_por_temp = []
for t in temps:
    validos = 0
    for _ in range(50):
        tune = A.extract_first_tune(gpt_best.generate(tok, max_new_tokens=MAX_NEW_TOKENS, temperature=t, top_k=TOP_K))
        if A.is_valid_abc(tune):
            validos += 1
    pct_por_temp.append(100.0 * validos / 50)

plt.figure(figsize=(6, 4))
plt.plot(temps, pct_por_temp, marker="o")
plt.xlabel("temperatura")
plt.ylabel("% tunes validos")
plt.title("Efecto de la temperatura (GPT)")
plt.grid(True)
plt.savefig("../informe/figs/ablation_temp.png", dpi=120)
plt.show()
print(dict(zip(temps, pct_por_temp)))